# Stable LR-LR-XGBoost Hybrid Model


In [ ]:
import json
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import cloudpickle
import cloudpickle
import joblib
import numpy as np
import pandas as pd
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import ADASYN, RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Configuration

Set `QUICK=True` only for a fast smoke test. For final reported results, keep `QUICK=False`.

In [ ]:
# The notebook uses the current notebook working folder as the project folder.
# If this .ipynb file is moved and opened from another folder, outputs will be saved there.
NOTEBOOK_DIR = Path.cwd()
ROOT = NOTEBOOK_DIR

# Put data.csv in the same folder as this notebook.
# If it is not found there, the notebook will search subfolders under the notebook folder.
DATA_FILENAME = "data.csv"


def find_dataset(data_filename: str = "data.csv") -> Path:
    candidates = [ROOT / data_filename]
    candidates.extend(sorted(ROOT.rglob(data_filename)))

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        resolved = candidate.resolve() if candidate.exists() else candidate.absolute()
        if resolved not in seen:
            seen.add(resolved)
            unique_candidates.append(candidate)

    existing = [candidate for candidate in unique_candidates if candidate.exists()]
    if not existing:
        raise FileNotFoundError(
            f"Could not find {data_filename}. Put the dataset file in the same folder as this notebook "
            "or in any subfolder under the notebook folder."
        )
    return existing[0]


DATA_PATH = find_dataset(DATA_FILENAME)

# OUTPUT_DIR is where the notebook saves results, not where it reads the dataset from.
# It is created under the current notebook folder.
# It will contain metrics_test.csv, classification_report.json, confusion_matrix_test.csv,
# run_config.json, and the saved trained model file.
OUTPUT_DIR = ROOT / "outputs" / "stable_lr_lrxgboost_notebook"

RANDOM_STATE = 42
QUICK = False

FINAL_WEIGHTS = [0.40, 0.20, 0.40]
FINAL_THRESHOLD = 0.425
TARGET_MACRO_F1 = 0.80
TARGET_DROPOUT_RECALL = 0.80

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Notebook working folder:", NOTEBOOK_DIR)
print("Data path:", DATA_PATH)
print("Output dir:", OUTPUT_DIR)


## 2. Strict S0 Feature Engineering

Only enrollment-stage features are used. First-semester, second-semester, and macroeconomic variables are excluded.

In [ ]:
S0_FEATURES = [
    "Marital status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Previous qualification (grade)",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Admission grade",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "Age at enrollment",
    "International",
]

FEATURE_VARIANTS = ["raw_s0", "basic_fe", "freq_fe", "interaction_fe", "all_fe"]
BASIC_NUMERIC_FEATURES = ["grade_gap"]
BASIC_CATEGORICAL_FEATURES = ["age_bin", "financial_risk", "support_signal"]
FREQ_SOURCE_FEATURES = [
    "Course",
    "Application mode",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
]
FREQ_NUMERIC_FEATURES = [f"{feature} frequency" for feature in FREQ_SOURCE_FEATURES]
INTERACTION_FEATURES = [
    "Course x Application mode",
    "Course x Scholarship holder",
    "Debtor x Tuition fees up to date",
]
CONTINUOUS_FEATURES = [
    "Previous qualification (grade)",
    "Admission grade",
    "Age at enrollment",
    "grade_gap",
    *FREQ_NUMERIC_FEATURES,
]


def uses_basic_features(variant: str) -> bool:
    return variant in {"basic_fe", "all_fe"}


def uses_frequency_features(variant: str) -> bool:
    return variant in {"freq_fe", "all_fe"}


def uses_interaction_features(variant: str) -> bool:
    return variant in {"interaction_fe", "all_fe"}


def engineered_feature_names(variant: str) -> list[str]:
    if variant not in FEATURE_VARIANTS:
        raise ValueError(f"Unknown feature variant: {variant}")
    features = list(S0_FEATURES)
    if uses_basic_features(variant):
        features.extend(BASIC_NUMERIC_FEATURES)
        features.extend(BASIC_CATEGORICAL_FEATURES)
    if uses_frequency_features(variant):
        features.extend(FREQ_NUMERIC_FEATURES)
    if uses_interaction_features(variant):
        features.extend(INTERACTION_FEATURES)
    forbidden = [feature for feature in features if feature.startswith("Curricular units")]
    if forbidden:
        raise ValueError(f"Forbidden curricular features generated: {forbidden}")
    return features


def clean_category(series: pd.Series) -> pd.Series:
    return series.astype("string").fillna("missing")


class ConservativeS0FeatureEngineer(BaseEstimator, TransformerMixin):
    """Create leakage-free, train-only S0 feature variants."""

    def __init__(self, variant: str = "raw_s0"):
        self.variant = variant

    def fit(self, x: pd.DataFrame, y: np.ndarray | None = None):
        if self.variant not in FEATURE_VARIANTS:
            raise ValueError(f"Unknown feature variant: {self.variant}")
        frame = pd.DataFrame(x).copy()
        missing = [feature for feature in S0_FEATURES if feature not in frame.columns]
        if missing:
            raise ValueError(f"Missing S0 features during FE fit: {missing}")

        self.frequency_maps_ = {}
        if uses_frequency_features(self.variant):
            for feature in FREQ_SOURCE_FEATURES:
                frequencies = clean_category(frame[feature]).value_counts(normalize=True)
                self.frequency_maps_[feature] = frequencies.to_dict()
        self.feature_names_out_ = engineered_feature_names(self.variant)
        return self

    def transform(self, x: pd.DataFrame) -> pd.DataFrame:
        frame = pd.DataFrame(x).copy()
        result = frame[S0_FEATURES].copy()

        if uses_basic_features(self.variant):
            result["grade_gap"] = result["Admission grade"] - result["Previous qualification (grade)"]
            result["age_bin"] = pd.cut(
                result["Age at enrollment"],
                bins=[-np.inf, 20, 24, 29, 39, np.inf],
                labels=["<=20", "21-24", "25-29", "30-39", "40+"],
            ).astype("string").fillna("missing")

            debtor = result["Debtor"].astype(int)
            tuition_unpaid = (result["Tuition fees up to date"].astype(int) == 0).astype(int)
            result["financial_risk"] = np.select(
                [
                    (debtor == 1) & (tuition_unpaid == 1),
                    (debtor == 1) & (tuition_unpaid == 0),
                    (debtor == 0) & (tuition_unpaid == 1),
                ],
                ["debtor_unpaid", "debtor_paid", "nodebt_unpaid"],
                default="nodebt_paid",
            )
            result["support_signal"] = (
                "scholarship_"
                + clean_category(result["Scholarship holder"])
                + "_displaced_"
                + clean_category(result["Displaced"])
                + "_special_"
                + clean_category(result["Educational special needs"])
            )

        if uses_frequency_features(self.variant):
            for feature in FREQ_SOURCE_FEATURES:
                mapping = self.frequency_maps_.get(feature, {})
                result[f"{feature} frequency"] = clean_category(result[feature]).map(mapping).fillna(0.0).astype(float)

        if uses_interaction_features(self.variant):
            result["Course x Application mode"] = clean_category(result["Course"]) + "__" + clean_category(
                result["Application mode"]
            )
            result["Course x Scholarship holder"] = clean_category(result["Course"]) + "__" + clean_category(
                result["Scholarship holder"]
            )
            result["Debtor x Tuition fees up to date"] = clean_category(result["Debtor"]) + "__" + clean_category(
                result["Tuition fees up to date"]
            )

        return result.loc[:, self.feature_names_out_]

    def get_feature_names_out(self, input_features=None) -> np.ndarray:
        return np.array(self.feature_names_out_, dtype=object)

## 3. Hybrid Model Classes

In [ ]:
def _classes_for(estimator: object) -> np.ndarray:
    if hasattr(estimator, "classes_"):
        return np.array(estimator.classes_)
    return np.array(estimator.named_steps["model"].classes_)


class WeightedProbabilityFusionClassifier(BaseEstimator, ClassifierMixin):
    """Blend already-probabilistic classifiers with fixed probability weights."""

    def __init__(self, estimators: list[tuple[str, object]], weights: list[float]):
        self.estimators = estimators
        self.weights = weights

    def fit(self, x: pd.DataFrame, y: np.ndarray):
        self.fitted_estimators_ = []
        for name, estimator in self.estimators:
            fitted = clone(estimator)
            fitted.fit(x, y)
            self.fitted_estimators_.append((name, fitted))
        self.classes_ = np.array(self.fitted_estimators_[0][1].classes_)
        weights = np.asarray(self.weights, dtype=float)
        self.weights_ = weights / weights.sum()
        return self

    def predict_proba(self, x: pd.DataFrame) -> np.ndarray:
        combined = None
        for weight, (_, estimator) in zip(self.weights_, self.fitted_estimators_):
            proba = estimator.predict_proba(x)
            estimator_classes = np.array(estimator.classes_)
            aligned = np.zeros((proba.shape[0], len(self.classes_)))
            for source_index, class_label in enumerate(estimator_classes):
                target_index = int(np.where(self.classes_ == class_label)[0][0])
                aligned[:, target_index] = proba[:, source_index]
            combined = aligned * weight if combined is None else combined + aligned * weight
        return combined

    def predict(self, x: pd.DataFrame) -> np.ndarray:
        return self.classes_[np.argmax(self.predict_proba(x), axis=1)]


class ThresholdTunedClassifier(BaseEstimator, ClassifierMixin):
    """Persist a fitted classifier with a binary dropout threshold."""

    def __init__(
        self,
        estimator: object,
        mode: str,
        dropout_index: int = 0,
        threshold: float = 0.5,
        multipliers: list[float] | None = None,
    ):
        self.estimator = estimator
        self.mode = mode
        self.dropout_index = dropout_index
        self.threshold = threshold
        self.multipliers = multipliers

    def fit(self, x: pd.DataFrame, y: np.ndarray):
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(x, y)
        self.classes_ = _classes_for(self.estimator_)
        return self

    def predict_proba(self, x: pd.DataFrame) -> np.ndarray:
        proba = self.estimator_.predict_proba(x)
        if self.mode != "multiclass_multiplier" or self.multipliers is None:
            return proba

        multipliers = np.asarray(self.multipliers, dtype=float)
        adjusted = proba * multipliers
        row_sums = adjusted.sum(axis=1, keepdims=True)
        return np.divide(adjusted, row_sums, out=np.zeros_like(adjusted), where=row_sums != 0)

    def predict(self, x: pd.DataFrame) -> np.ndarray:
        proba = self.predict_proba(x)
        if self.mode == "binary_threshold":
            other_indices = [index for index in range(proba.shape[1]) if index != self.dropout_index]
            other_index = other_indices[0]
            return np.where(proba[:, self.dropout_index] >= self.threshold, self.dropout_index, other_index)
        return self.classes_[np.argmax(proba, axis=1)]

## 4. Data Loading and Pipeline Builders

In [ ]:
def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";")
    df.columns = [column.strip() for column in df.columns]
    if df.shape != (4424, 37):
        raise ValueError(f"Unexpected dataset shape: {df.shape}")
    if any(feature.startswith("Curricular units") for feature in S0_FEATURES):
        raise ValueError("S0 raw features must not include curricular units.")
    return df


def prepare_binary(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, list[str], dict[str, str]]:
    task_df = df[df["Target"].isin(["Dropout", "Graduate"])].copy()
    encoder = LabelEncoder()
    y = encoder.fit_transform(task_df["Target"])
    labels = list(encoder.classes_)
    if labels != ["Dropout", "Graduate"]:
        raise ValueError(f"Unexpected binary labels: {labels}")
    return task_df[S0_FEATURES].copy(), y, labels, {str(index): label for index, label in enumerate(labels)}


def split_data(x: pd.DataFrame, y: np.ndarray, random_state: int):
    x_train_val, x_test, y_train_val, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        stratify=y,
        random_state=random_state,
    )
    x_train, x_val, y_train, y_val = train_test_split(
        x_train_val,
        y_train_val,
        test_size=0.25,
        stratify=y_train_val,
        random_state=random_state,
    )
    return x_train, x_val, x_test, y_train, y_val, y_test


def make_preprocessor(variant: str) -> ColumnTransformer:
    features = engineered_feature_names(variant)
    numeric = [feature for feature in features if feature in CONTINUOUS_FEATURES]
    categorical = [feature for feature in features if feature not in numeric]
    return ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical),
        ],
        remainder="drop",
    )


def sampler_step(name: str, random_state: int):
    if name == "none":
        return None
    if name == "ros":
        return RandomOverSampler(random_state=random_state)
    if name == "adasyn":
        return ADASYN(random_state=random_state)
    if name == "smoteenn":
        return SMOTEENN(random_state=random_state)
    raise ValueError(f"Unknown sampler: {name}")


def make_pipeline(estimator: object, variant: str, sampler: str, random_state: int) -> ImbPipeline:
    steps = [
        ("feature_engineering", ConservativeS0FeatureEngineer(variant)),
        ("preprocess", make_preprocessor(variant)),
    ]
    sampler_obj = sampler_step(sampler, random_state)
    if sampler_obj is not None:
        steps.append(("sampler", sampler_obj))
    steps.append(("model", estimator))
    return ImbPipeline(steps=steps)


def make_xgb(random_state: int, quick: bool, conservative: bool = False) -> XGBClassifier:
    return XGBClassifier(
        n_estimators=160 if quick else 320,
        learning_rate=0.035 if conservative else 0.05,
        max_depth=2 if conservative else 3,
        min_child_weight=3 if conservative else 1,
        subsample=0.9,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=5.0 if conservative else 2.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=random_state,
        n_jobs=-1,
    )


def lr_pipeline(variant: str, sampler: str, random_state: int, c_value: float = 1.0):
    return make_pipeline(
        LogisticRegression(C=c_value, class_weight="balanced", max_iter=5000, random_state=random_state),
        variant,
        sampler,
        random_state,
    )


def xgb_pipeline(variant: str, sampler: str, random_state: int, quick: bool, conservative: bool = True):
    return make_pipeline(make_xgb(random_state, quick, conservative=conservative), variant, sampler, random_state)


def predict_threshold(proba: np.ndarray, dropout_index: int, threshold: float) -> np.ndarray:
    other_index = 1 - dropout_index
    return np.where(proba[:, dropout_index] >= threshold, dropout_index, other_index)

## 5. Final Proposed Stable LR-LR-XGBoost Hybrid

This cell trains only the final proposed model. It does not redo the full repeated-CV search.

In [ ]:
def build_stable_lr_lrxgboost_model(random_state: int, quick: bool = False) -> ThresholdTunedClassifier:
    lr_basic_none = lr_pipeline("basic_fe", "none", random_state)
    lr_freq_adasyn = lr_pipeline("freq_fe", "adasyn", random_state)
    xgb_freq_adasyn = xgb_pipeline("freq_fe", "adasyn", random_state, quick, conservative=True)

    fusion = WeightedProbabilityFusionClassifier(
        estimators=[
            ("lr_basic_none", lr_basic_none),
            ("lr_freq_adasyn", lr_freq_adasyn),
            ("xgb_freq_adasyn", xgb_freq_adasyn),
        ],
        weights=FINAL_WEIGHTS,
    )
    return ThresholdTunedClassifier(
        estimator=fusion,
        mode="binary_threshold",
        dropout_index=0,
        threshold=FINAL_THRESHOLD,
    )


def compute_metrics(y_true: np.ndarray, pred: np.ndarray, proba: np.ndarray) -> dict[str, float]:
    macro_f1 = f1_score(y_true, pred, average="macro", zero_division=0)
    dropout_recall = recall_score(y_true == 0, pred == 0, zero_division=0)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision_macro": precision_score(y_true, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, pred, average="macro", zero_division=0),
        "f1_macro": macro_f1,
        "f1_weighted": f1_score(y_true, pred, average="weighted", zero_division=0),
        "precision_Dropout": precision_score(y_true == 0, pred == 0, zero_division=0),
        "recall_Dropout": dropout_recall,
        "f1_Dropout": f1_score(y_true == 0, pred == 0, zero_division=0),
        "roc_auc_dropout": roc_auc_score(y_true == 0, proba[:, 0]),
    }


def evaluate_and_save(model, x_test, y_test, labels, output_dir: Path):
    proba = model.predict_proba(x_test)
    pred = model.predict(x_test)
    metrics = compute_metrics(y_test, pred, proba)

    metrics_df = pd.DataFrame([
        {
            "split": "test",
            "model": "Stable LR-LR-XGBoost Hybrid",
            "weights": "/".join(f"{weight:.2f}" for weight in FINAL_WEIGHTS),
            "threshold": FINAL_THRESHOLD,
            **metrics,
        }
    ])
    metrics_df.to_csv(output_dir / "metrics_test.csv", index=False)

    report = classification_report(
        y_test,
        pred,
        labels=list(range(len(labels))),
        target_names=labels,
        output_dict=True,
        zero_division=0,
    )
    (output_dir / "classification_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

    matrix = confusion_matrix(y_test, pred, labels=list(range(len(labels))))
    pd.DataFrame(
        matrix,
        index=[f"actual_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    ).to_csv(output_dir / "confusion_matrix_test.csv")

    with open(output_dir / "stable_lr_lrxgboost_hybrid.pkl", "wb") as f:
        cloudpickle.dump(model, f)
    return metrics_df, report

## 6. Run Final Model

The final model is trained on the combined train + validation data and evaluated once on the hold-out test set, matching the paper setting.

In [ ]:
df = load_data(DATA_PATH)
x, y, labels, label_map = prepare_binary(df)
x_train, x_val, x_test, y_train, y_val, y_test = split_data(x, y, RANDOM_STATE)
x_train_val = pd.concat([x_train, x_val], axis=0)
y_train_val = np.concatenate([y_train, y_val])

for variant in ["basic_fe", "freq_fe"]:
    forbidden = [feature for feature in engineered_feature_names(variant) if feature.startswith("Curricular units")]
    if forbidden:
        raise ValueError(f"Feature leakage detected in {variant}: {forbidden}")

start = time.perf_counter()
model = build_stable_lr_lrxgboost_model(RANDOM_STATE, QUICK)
model.fit(x_train_val, y_train_val)
fit_seconds = time.perf_counter() - start

metrics_df, report = evaluate_and_save(model, x_test, y_test, labels, OUTPUT_DIR)

run_config = {
    "task": "binary Dropout vs Graduate",
    "feature_set": "strict_s0",
    "raw_s0_features": S0_FEATURES,
    "label_map": label_map,
    "model": "Stable LR-LR-XGBoost Hybrid",
    "weights": FINAL_WEIGHTS,
    "threshold": FINAL_THRESHOLD,
    "fit_seconds": fit_seconds,
    "quick": QUICK,
    "output_dir": str(OUTPUT_DIR),
}
(OUTPUT_DIR / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print(f"Fit seconds: {fit_seconds:.2f}")
display(metrics_df)
print("Saved outputs to:", OUTPUT_DIR)

## 7. Optional: Full Repeated-CV Search

The paper's selected model came from repeated stratified CV. The cell below is optional and can take longer. Set `RUN_REPEATED_CV=True` to run it.

In [ ]:
RUN_REPEATED_CV = False
FOLDS = 5
REPEATS = 3

V1 = {
    "model": "Proposed LR-XGBoost Hybrid v1",
    "accuracy": 0.78099173553719,
    "macro_f1": 0.7751520782567537,
    "dropout_recall": 0.7922535211267606,
    "weighted_f1": 0.7830381111653772,
    "roc_auc_dropout": 0.8603020839971958,
}

@dataclass(frozen=True)
class Candidate:
    name: str
    estimator: object
    family: str


def threshold_grid(quick: bool) -> np.ndarray:
    return np.round(np.arange(0.35, 0.5201, 0.01 if quick else 0.005), 3)


def candidate_models(random_state: int, quick: bool) -> list[Candidate]:
    candidates: list[Candidate] = []
    lr_basic_none = lr_pipeline("basic_fe", "none", random_state)
    lr_basic_ros = lr_pipeline("basic_fe", "ros", random_state)
    lr_freq_none = lr_pipeline("freq_fe", "none", random_state)
    lr_freq_adasyn = lr_pipeline("freq_fe", "adasyn", random_state)
    xgb_freq_adasyn = xgb_pipeline("freq_fe", "adasyn", random_state, quick, conservative=True)
    xgb_freq_none = xgb_pipeline("freq_fe", "none", random_state, quick, conservative=True)

    for lr_weight in [0.35, 0.45, 0.55]:
        candidates.append(
            Candidate(
                name=f"V1_LR_XGBcons_freqADASYN_lr{lr_weight:.2f}",
                estimator=WeightedProbabilityFusionClassifier(
                    estimators=[("lr_freq_adasyn", lr_freq_adasyn), ("xgb_freq_adasyn", xgb_freq_adasyn)],
                    weights=[lr_weight, 1.0 - lr_weight],
                ),
                family="v1_lr_xgb",
            )
        )

    for lr_weight in [0.30, 0.40, 0.50]:
        candidates.append(
            Candidate(
                name=f"CrossFE_LRbasic_XGBfreqADASYN_lr{lr_weight:.2f}",
                estimator=WeightedProbabilityFusionClassifier(
                    estimators=[("lr_basic_none", lr_basic_none), ("xgb_freq_adasyn", xgb_freq_adasyn)],
                    weights=[lr_weight, 1.0 - lr_weight],
                ),
                family="cross_feature",
            )
        )
        candidates.append(
            Candidate(
                name=f"CrossFE_LRbasicROS_XGBfreqADASYN_lr{lr_weight:.2f}",
                estimator=WeightedProbabilityFusionClassifier(
                    estimators=[("lr_basic_ros", lr_basic_ros), ("xgb_freq_adasyn", xgb_freq_adasyn)],
                    weights=[lr_weight, 1.0 - lr_weight],
                ),
                family="cross_feature",
            )
        )

    for weights in [(0.30, 0.20, 0.50), (0.35, 0.15, 0.50), (0.40, 0.20, 0.40)]:
        candidates.append(
            Candidate(
                name=f"Tri_LRbasic_LRfreq_XGBfreq_{weights[0]:.2f}_{weights[1]:.2f}_{weights[2]:.2f}",
                estimator=WeightedProbabilityFusionClassifier(
                    estimators=[
                        ("lr_basic_none", lr_basic_none),
                        ("lr_freq_adasyn", lr_freq_adasyn),
                        ("xgb_freq_adasyn", xgb_freq_adasyn),
                    ],
                    weights=list(weights),
                ),
                family="tri_blend",
            )
        )

    candidates.append(
        Candidate(
            name="DualXGB_freqADASYN_freqNone_w0.70",
            estimator=WeightedProbabilityFusionClassifier(
                estimators=[("xgb_freq_adasyn", xgb_freq_adasyn), ("xgb_freq_none", xgb_freq_none)],
                weights=[0.70, 0.30],
            ),
            family="xgb_blend",
        )
    )
    candidates.append(
        Candidate(
            name="LR_ensemble_basicROS_freqNone_freqADASYN",
            estimator=WeightedProbabilityFusionClassifier(
                estimators=[("lr_basic_ros", lr_basic_ros), ("lr_freq_none", lr_freq_none), ("lr_freq_adasyn", lr_freq_adasyn)],
                weights=[0.50, 0.25, 0.25],
            ),
            family="lr_blend",
        )
    )
    return candidates


def repeated_cv_rows(x_train_val, y_train_val, candidates, folds, repeats, thresholds, random_state):
    splitter = RepeatedStratifiedKFold(n_splits=folds, n_repeats=repeats, random_state=random_state)
    rows = []
    splits = list(splitter.split(x_train_val, y_train_val))
    for candidate in candidates:
        print(f"Repeated-CV candidate: {candidate.name}")
        for split_index, (train_index, valid_index) in enumerate(splits, start=1):
            fold_model = clone(candidate.estimator)
            start = time.perf_counter()
            fold_model.fit(x_train_val.iloc[train_index], y_train_val[train_index])
            fit_seconds = time.perf_counter() - start
            proba = fold_model.predict_proba(x_train_val.iloc[valid_index])
            y_valid = y_train_val[valid_index]
            for threshold in thresholds:
                pred = predict_threshold(proba, 0, float(threshold))
                metric = compute_metrics(y_valid, pred, proba)
                macro_f1 = metric["f1_macro"]
                dropout_recall = metric["recall_Dropout"]
                metric.update({
                    "combined_50m50r": 0.5 * macro_f1 + 0.5 * dropout_recall,
                    "combined_30m70r": 0.3 * macro_f1 + 0.7 * dropout_recall,
                    "target_080_distance": ((max(0.0, TARGET_MACRO_F1 - macro_f1)) ** 2 + (max(0.0, TARGET_DROPOUT_RECALL - dropout_recall)) ** 2) ** 0.5,
                    "target_080_min_margin": min(macro_f1 - TARGET_MACRO_F1, dropout_recall - TARGET_DROPOUT_RECALL),
                    "margin_vs_v1_macro_f1": macro_f1 - V1["macro_f1"],
                    "margin_vs_v1_dropout_recall": dropout_recall - V1["dropout_recall"],
                })
                rows.append({
                    "model": candidate.name,
                    "family": candidate.family,
                    "split_index": split_index,
                    "threshold": float(threshold),
                    "fit_seconds": fit_seconds,
                    **metric,
                })
    return rows


def summarize_cv(fold_df: pd.DataFrame) -> pd.DataFrame:
    metric_columns = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted",
        "precision_Dropout", "recall_Dropout", "f1_Dropout", "roc_auc_dropout",
        "combined_50m50r", "combined_30m70r", "target_080_distance", "target_080_min_margin",
        "margin_vs_v1_macro_f1", "margin_vs_v1_dropout_recall",
    ]
    summary = fold_df.groupby(["family", "model", "threshold"])[metric_columns].agg(["mean", "std", "min"])
    summary.columns = ["_".join(column).strip("_") for column in summary.columns]
    summary = summary.reset_index()
    summary["stable_score"] = (
        0.45 * summary["f1_macro_mean"]
        + 0.45 * summary["recall_Dropout_mean"]
        + 0.10 * summary["f1_Dropout_mean"]
        - 0.50 * summary["f1_macro_std"].fillna(0.0)
        - 0.50 * summary["recall_Dropout_std"].fillna(0.0)
    )
    summary["lcb_macro_f1"] = summary["f1_macro_mean"] - summary["f1_macro_std"].fillna(0.0)
    summary["lcb_dropout_recall"] = summary["recall_Dropout_mean"] - summary["recall_Dropout_std"].fillna(0.0)
    summary["lcb_min_080_margin"] = np.minimum(
        summary["lcb_macro_f1"] - TARGET_MACRO_F1,
        summary["lcb_dropout_recall"] - TARGET_DROPOUT_RECALL,
    )
    return summary


if RUN_REPEATED_CV:
    thresholds = threshold_grid(QUICK)
    candidates = candidate_models(RANDOM_STATE, QUICK)
    fold_rows = repeated_cv_rows(
        x_train_val=x_train_val,
        y_train_val=y_train_val,
        candidates=candidates,
        folds=FOLDS,
        repeats=REPEATS,
        thresholds=thresholds,
        random_state=RANDOM_STATE,
    )
    fold_df = pd.DataFrame(fold_rows)
    summary_df = summarize_cv(fold_df)
    fold_df.to_csv(OUTPUT_DIR / "fold_metrics_cv.csv", index=False)
    summary_df.sort_values(["stable_score", "combined_50m50r_mean"], ascending=False).to_csv(
        OUTPUT_DIR / "summary_cv.csv", index=False
    )
    display(summary_df.sort_values(["stable_score", "combined_50m50r_mean"], ascending=False).head(10))
else:
    print("Skipped repeated-CV search. Set RUN_REPEATED_CV=True to run it.")